In [ ]:
!pip install admet_ai


In [ ]:
import json
import torch
import numpy as np
from argparse import Namespace
from admet_ai import ADMETModel
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen

torch.serialization.add_safe_globals([
    Namespace,
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype,
    np.dtypes.Float64DType
])

# ── Load docking results from QAOA notebook ──────────────────────────────────
# The QAOA notebook writes /content/docking_results.json after every ligand run.
# Edit RESULTS_PATH if you saved the file elsewhere (e.g. mounted Google Drive).
RESULTS_PATH = '/content/docking_results.json'

with open(RESULTS_PATH) as _f:
    _results = json.load(_f)

binding_scores      = _results["binding_scores"]
ligand_smiles_list  = _results["ligand_smiles_list"]
print(f"Loaded {len(binding_scores)} docking results from {RESULTS_PATH}")

# ── ADMET prediction ──────────────────────────────────────────────────────────
model = ADMETModel()
admet_df = model.predict(smiles=ligand_smiles_list)
admet_df = admet_df.reset_index().rename(columns={'index': 'SMILES'})

def lipinski_ok(mol):
    return (Descriptors.MolWt(mol) < 500 and
            Crippen.MolLogP(mol) < 5 and
            Lipinski.NumHDonors(mol) <= 5 and
            Lipinski.NumHAcceptors(mol) <= 10)

lipinski_flags = [lipinski_ok(Chem.MolFromSmiles(smi)) for smi in admet_df['SMILES']]
admet_df["lipinski_pass"] = lipinski_flags

good_admet = (
    (admet_df["HIA_Hou"]      == True)  &
    (admet_df["BBB_Martins"]  == False) &
    (admet_df["hERG"]         == False) &
    (admet_df["CYP3A4_Veith"] == False) &
    (admet_df["lipinski_pass"])
)
admet_df["admet_pass"] = good_admet

# ── Combine with binding scores and rank ──────────────────────────────────────
rank_df = (
    admet_df
    .assign(binding_score=binding_scores)
    .sort_values(["admet_pass", "binding_score"], ascending=[False, False])
    .reset_index(drop=True)
)

print("--- DETAILED ADMET PROFILE ---")
print(rank_df[[
    "SMILES",
    "binding_score",
    "HIA_Hou",
    "BBB_Martins",
    "hERG",
    "CYP3A4_Veith",
    "lipinski_pass",
    "admet_pass"
]].head(10))